In [18]:
from langchain_community.document_loaders.csv_loader import CSVLoader

hyperbook_docs = CSVLoader(
    file_path='../../data/raw/hyperbook.csv', 
    content_columns=['content'], 
    metadata_columns=['url', 'title', 'authors'],
    encoding='utf-8'
).load()

In [19]:
from pathlib import Path
from langchain_community.document_loaders.pdf import PyPDFLoader

textbooks_docs = []

ball_metadata = {
    'url': 'https://mcgill.on.worldcat.org/oclc/180470082',
    'title': "Shapes: Nature's patterns: a tapestry in three parts",
    'authors': 'Philip Ball'
}

def set_metadata(doc):
    doc.metadata = ball_metadata
    return doc

# Temporary — Only experimenting with three chapters from Shapes by Ball, P.
for chapter_path in Path('../../data/raw/textbooks/shapes_ball').iterdir():
    chapter_docs = PyPDFLoader(chapter_path).load()
    chapter_docs = list(map(set_metadata, chapter_docs))
    textbooks_docs = textbooks_docs + chapter_docs 

In [20]:
docs = hyperbook_docs + textbooks_docs

In [21]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Chunk parameters
chunk_size = 256
chunk_overlap = 32

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
doc_splits = text_splitter.split_documents(docs)

In [22]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

embeddings = HuggingFaceEmbeddings(model_name="Qwen/Qwen3-Embedding-0.6B")
vectorstore = InMemoryVectorStore.from_documents(documents=doc_splits, embedding=embeddings)
vectorstore.dump(f'../../embeddings/bien200_embeddings')

In [23]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_ollama import OllamaEmbeddings

# Chunk parameters
chunk_size, chunk_overlap = 256, 32
embedding_model = OllamaEmbeddings(model='dengcao/Qwen3-Embedding-0.6B:Q8_0')
embeddings_path = f'../../embeddings/bien200_embeddings'

# Loading the vector store locally
vectorstore = InMemoryVectorStore.load(path=embeddings_path, embedding=embedding_model)
dense_retriever = vectorstore.as_retriever()

In [25]:
dense_retriever.invoke('Shapes in nature')

[Document(id='00b12411-bb4b-4e81-9a61-a2fa38dd28f2', metadata={'url': 'https://mcgill.on.worldcat.org/oclc/180470082', 'title': "Shapes: Nature's patterns: a tapestry in three parts", 'authors': 'Philip Ball'}, page_content='Fig. 1.5: Chemistry alone can also produce microscopic surface patterns rather similar to those seen \nin meteorite ALH84001 ( ). (Photo: Geoffrey Ozin, University of Toronto.)Fig. 1.3\nWhat is it that encourages us to typecast some forms as those made by life, and others as the \nproducts of the non-living world? A tree, a rabbit, a spider have rather little in common when \nconsidered as mere shapes—and yet we don’t hesitate to see them as examples of living morphology. \nWhy? Perhaps we sense a kind of purpose, of design, in these forms? They are ‘complex’, certainly, \nbut what does that mean? They may have some regularity or symmetry—the bilateral symmetry of \nthe animals, the repeated branching of the tree—but that can’t be all there is to it, for surely it 